In [7]:
import cv2
import ffmpeg
import ipywidgets as widgets
import json
import mediapipe as mp
import numpy as np
import pyttsx3
import re
import sounddevice as sd
import threading
import time
import vosk
from typing import Any, Dict, cast

from datetime import datetime
from IPython.display import display
from pathlib import Path
from collections import deque



In [8]:
# =========================================================
# Konfiguracja
# =========================================================
MODEL_PATH = "pose_landmarker_full.task"
VOSK_MODEL_PATH = "vosk-model-small-pl-0.22"

FRONT_STREAM_URL = 0
SIDE_STREAM_URL = 1

STATE_CONFIRM_FRAMES = 3
MIN_REP_INTERVAL = 1.0

ANGLE_WINDOW = 5
angle_buffer = deque(maxlen=ANGLE_WINDOW)

MAX_WIDTH_PER_CAMERA = 640
JPEG_QUALITY = 65
FPS = 40
TIMEOUT = 0

HYSTERESIS = 8

vosk.SetLogLevel(-1)

speech_lock = threading.Lock()
AUTO_SAVE_TO_FILE = True
AUTO_START_SERIES = True
BOTH_CAMERAS_REQUIRED = False

BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

POSE_CONNECTIONS = [
	(0, 1), (1, 2), (2, 3), (3, 7), (0, 4), (4, 5), (5, 6), (6, 8),
	(9, 10), (11, 12), (11, 23), (12, 24), (23, 24),
	(11, 13), (13, 15), (15, 17), (15, 19), (15, 21), (17, 19),
	(12, 14), (14, 16), (16, 18), (16, 20), (16, 22), (18, 20),
	(23, 25), (25, 27), (27, 29), (29, 31),
	(24, 26), (26, 28), (28, 30), (30, 32),
	(27, 31), (28, 32),
]

EXERCISE_DEFINITIONS: Dict[str, Dict[str, Any]] = {
	"dumbbell_curl": {
		"label": "Podnoszenie hantli",
		"rep_rule": "Pelny ruch: lokiec < 40 stopni i > 140 stopni",
		"down_threshold": 40,
		"up_threshold": 140,
		"min_amplitude": 100,
		"symmetry_threshold": 15,
		"sway_threshold": 15,
		"elbow_threshold": 25,
		"notes": [
			"Kamera frontowa sprawdza symetrie ramion.",
			"Kamera boczna stabilizuje liczenie faz ruchu.",
			"Komendy: start / seria / reset / stop / zapis on / zapis off.",
			"W sessions/ zapisywany jest film i JSON dla kazdej serii.",
		],
	}
}
ACTIVE_EXERCISE_KEY = "dumbbell_curl"
VOICE_COMMANDS = ["start", "seria", "reset", "stop", "zapis on", "zapis off"]

START_WORDS = ["start", "zaczynam", "zacznij", "zacznijmy", "startujemy"]
STOP_WORDS = ["stop", "koniec", "zatrzymaj", "skończ", "skoncz", "koncz"]
SERIES_WORDS = ["seria", "serie", "serię", "serii"]

vosk_model = vosk.Model(VOSK_MODEL_PATH)

stop = threading.Event()
state_lock = threading.Lock()

import queue
tts_queue = queue.Queue()

latest_frames: Dict[str, Any] = {"front": None, "side": None}
latest_landmarks: Dict[str, Any] = {"front": None, "side": None}

# Zmienne do eliminacji duplikatów klatek (Deduplikacja procesora)
latest_timestamps = {"front": 0, "side": 0}
processed_timestamps = {"front": 0, "side": 0}

dumbbell_state: Dict[str, Any] = {
	"phase": "extended",  # "extended" (wyprost) lub "flexed" (ugięcie)
	"reps": 0,
	"cycle_min": 180.0,
	"cycle_max": 0.0,
	"amplitude_warned": False,
	"symmetry_warned": False,
	"sway_warned": False,
	"elbow_warned": False,
	"candidate_phase": None,
	"candidate_count": 0,
	"last_rep_time": 0
}

recording: Dict[str, Any] = {
	"active": False,
	"frames": [],
	"video_writer": None,
	"video_path": None,
	"stats_by_rep": {},  # Nowa ustrukturyzowana forma slownika pod JSON
	"series_index": 1,
	"started_at": None,
}

save_cfg: Dict[str, Any] = {"enabled": AUTO_SAVE_TO_FILE}

img_widget = widgets.Image(format="jpeg", layout=widgets.Layout(
	border="2px solid #1a1a2e",
	border_radius="12px",
	width="100%",
))
status_html = widgets.HTML(value="")
stats_html = widgets.HTML(value="")
voice_html = widgets.HTML(value="")
assumptions_html = widgets.HTML(value="")


def get_active_exercise():
	return EXERCISE_DEFINITIONS[ACTIVE_EXERCISE_KEY]

def tts_worker():
	import pyttsx3
	try:
		engine = pyttsx3.init()
		engine.setProperty("rate", 160)
		while not stop.is_set():
			try:
				text = tts_queue.get(timeout=0.2)
				engine.say(text)
				engine.runAndWait()
				tts_queue.task_done()
			except queue.Empty:
				continue
	except Exception as e:
		print(f"[TTS Error] Nie udało się przetworzyć mowy w workerze: {e}")


def speak(text):
	tts_queue.put(text)


def angle(a, b, c):
	pa = np.array([a.x, a.y])
	pb = np.array([b.x, b.y])
	pc = np.array([c.x, c.y])
	ba = pa - pb
	bc = pc - pb
	den = (np.linalg.norm(ba) * np.linalg.norm(bc))
	if den == 0:
		return 180.0
	cosv = np.dot(ba, bc) / den
	return float(np.degrees(np.arccos(np.clip(cosv, -1.0, 1.0))))


def arm_angles(landmarks):
	left = angle(landmarks[11], landmarks[13], landmarks[15])
	right = angle(landmarks[12], landmarks[14], landmarks[16])
	return left, right, (left + right) / 2.0


def draw_pose(frame, landmarks, line_color=(0, 255, 136), point_color=(255, 100, 100)):
	h, w = frame.shape[:2]
	for start_idx, end_idx in POSE_CONNECTIONS:
		p1 = landmarks[start_idx]
		p2 = landmarks[end_idx]
		cv2.line(frame, (int(p1.x * w), int(p1.y * h)), (int(p2.x * w), int(p2.y * h)), line_color, 2)
	for lm in landmarks:
		cv2.circle(frame, (int(lm.x * w), int(lm.y * h)), 4, point_color, -1)


def resize_frame(frame, max_width):
	h, w = frame.shape[:2]
	if w <= max_width:
		return frame
	return cv2.resize(frame, (max_width, int(h * max_width / w)))


def compose_frames(front_frame, side_frame):
	if front_frame is None and side_frame is None:
		return None

	if front_frame is None:
		front_frame = np.zeros((480, MAX_WIDTH_PER_CAMERA, 3), dtype=np.uint8)
		cv2.putText(front_frame, "BRAK FRONT", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
	if side_frame is None:
		side_frame = np.zeros((480, MAX_WIDTH_PER_CAMERA, 3), dtype=np.uint8)
		cv2.putText(side_frame, "BRAK BOK", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

	front_frame = resize_frame(front_frame, MAX_WIDTH_PER_CAMERA)
	side_frame = resize_frame(side_frame, MAX_WIDTH_PER_CAMERA)

	target_h = int(max(front_frame.shape[0], side_frame.shape[0]))
	if front_frame.shape[0] != target_h:
		front_frame = cv2.resize(front_frame, (int(front_frame.shape[1]), int(target_h)))
	if side_frame.shape[0] != target_h:
		side_frame = cv2.resize(side_frame, (int(side_frame.shape[1]), int(target_h)))

	cv2.putText(front_frame, "FRONT", (16, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
	cv2.putText(side_frame, "BOK", (16, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
	return np.hstack([front_frame, side_frame])


def render_status(active):
	dot = "#00ff88" if active else "#ff4444"
	label = "● NAGRYWANIE" if active else "● CZUWANIE"
	save_badge = "AUTO-ZAPIS ON" if save_cfg["enabled"] else "AUTO-ZAPIS OFF"
	status_html.value = f"""
	<div style='font-family:monospace;font-size:13px;color:{dot};
	background:#0d0d1a;padding:6px 14px;border-radius:8px;
	display:inline-block;letter-spacing:1px;margin-bottom:6px'>{label} | {save_badge}</div>"""


def render_voice(partial, final=""):
	voice_html.value = f"""
	<div style='font-family:monospace;font-size:12px;background:#0d0d1a;
	padding:6px 14px;border-radius:8px;margin-top:4px;border:1px solid #1e1e3a'>
		<span style='color:#555'>MIC:</span>
		<span style='color:#aaa'>{partial}</span>
		{"<span style='color:#00ff88;margin-left:8px'>-> " + final + "</span>" if final else ""}
	</div>"""


def render_assumptions():
	exercise = get_active_exercise()
	notes_html = "".join(f"<li>{note}</li>" for note in list(exercise["notes"]))
	commands_html = ", ".join(f"<code>{cmd}</code>" for cmd in VOICE_COMMANDS)
	assumptions_html.value = f"""
	<div style='font-family:monospace;background:#101322;border-radius:12px;padding:12px 18px;margin-bottom:6px;border:1px solid #1e1e3a'>
		<div style='color:#888;font-size:11px;letter-spacing:1px;margin-bottom:4px'>AKTYWNY SCENARIUSZ</div>
		<div style='color:#fff;font-size:16px;font-weight:bold;margin-bottom:6px'>{exercise['label']}</div>
		<div style='color:#bbb;font-size:12px;margin-bottom:6px'>{exercise['rep_rule']}</div>
		<ul style='color:#ddd;font-size:12px;margin:0 0 6px 18px;padding:0'>{notes_html}</ul>
		<div style='color:#888;font-size:11px'>Komendy: {commands_html}</div>
	</div>"""


def render_stats(reps, avg_angle, good, reps_dict, source_label):
	exercise = get_active_exercise()

	# Wyciągamy 3 ostatnio zarejestrowane błędy z ustrukturyzowanego słownika
	recent_errors = []
	for r_key, r_val in reps_dict.items():
		rep_num = r_key.split("_")[-1]
		for err in r_val.get("errors", []):
			recent_errors.append({"rep": rep_num, "error": err["error"], "value": err["value"]})

	err_html = "".join([
		f"<div style='color:#ff6b6b;font-size:12px;margin-top:3px'>pow. {e['rep']}: {e['error']} ({e['value']})</div>"
		for e in recent_errors[-3:]
	])

	color = "#00ff88" if good else "#ff6b6b"
	phase_icon = dumbbell_state["phase"].upper() # Bezpieczne czytanie zmiennej zamiast wywoływania confirm_phase!

	stats_html.value = f"""
	<div style='font-family:monospace;background:#0d0d1a;border-radius:12px;padding:12px 18px;margin-top:6px;border:1px solid #1e1e3a'>
		<div style='display:flex;gap:24px;align-items:center;flex-wrap:wrap'>
			<div>
				<div style='color:#888;font-size:11px;letter-spacing:1px'>CWICZENIE</div>
				<div style='color:#fff;font-size:18px;font-weight:bold'>{exercise['label']}</div>
			</div>
			<div>
				<div style='color:#888;font-size:11px;letter-spacing:1px'>POWTORZENIA</div>
				<div style='color:#fff;font-size:28px;font-weight:bold'>{reps}</div>
			</div>
			<div>
				<div style='color:#888;font-size:11px;letter-spacing:1px'>KAT LOKCIA</div>
				<div style='color:{color};font-size:28px;font-weight:bold'>{avg_angle:.0f}</div>
			</div>
			<div>
				<div style='color:#888;font-size:11px;letter-spacing:1px'>FAZA</div>
				<div style='color:#aaa;font-size:20px'>{phase_icon}</div>
			</div>
			<div>
				<div style='color:#888;font-size:11px;letter-spacing:1px'>ANALIZA</div>
				<div style='color:#aaa;font-size:14px'>{source_label}</div>
			</div>
			<div>
				<div style='color:#888;font-size:11px;letter-spacing:1px'>FORMA</div>
				<div style='color:{color};font-size:16px;font-weight:bold'>{'OK' if good else 'DOKONCZ RUCH'}</div>
			</div>
		</div>
		{err_html}
	</div>"""


def reset_motion_state(reset_reps=True):
	dumbbell_state["phase"] = "extended"
	dumbbell_state["cycle_min"] = 180.0
	dumbbell_state["cycle_max"] = 0.0
	dumbbell_state["amplitude_warned"] = False
	dumbbell_state["symmetry_warned"] = False
	dumbbell_state["sway_warned"] = False
	dumbbell_state["elbow_warned"] = False
	dumbbell_state["candidate_phase"] = None
	dumbbell_state["candidate_count"] = 0
	if reset_reps:
		dumbbell_state["reps"] = 0


def analyze_dumbbell_curl(front_landmarks, side_landmarks):
	exercise = get_active_exercise()
	front_avg = None
	side_avg = None
	try:
		if front_landmarks is not None:
			fl, fr, front_avg = arm_angles(front_landmarks)
		if side_landmarks is not None:
			sl, sr, side_avg = arm_angles(side_landmarks)
	except Exception:
		pass

	if front_avg is None and side_avg is None:
		return dumbbell_state["reps"], 0.0, True, "brak sylwetki"

	if front_avg is not None and side_avg is not None:
		raw_angle = (front_avg + side_avg) / 2
		angle_buffer.append(raw_angle)
		avg_angle = sum(angle_buffer) / len(angle_buffer)
		both_available = True
	else:
		avg_angle = side_avg if side_avg is not None else front_avg
		both_available = False

	if BOTH_CAMERAS_REQUIRED and not both_available:
		return dumbbell_state["reps"], float(avg_angle if avg_angle is not None else 0.0), True, "oczekuje na obie kamery"

	dumbbell_state["cycle_min"] = min(dumbbell_state.get("cycle_min", 180.0), float(avg_angle))
	dumbbell_state["cycle_max"] = max(dumbbell_state.get("cycle_max", 0.0), float(avg_angle))

	down_thresh = exercise["down_threshold"]  # 40 (maksymalne ugięcie ramienia)
	up_thresh = exercise["up_threshold"]      # 140 (pełny wyprost)

	# --- NOWA ODILZOLOWANA MASZYNA STANÓW (Zamiast popsutego confirm_phase) ---
	if avg_angle < (down_thresh - HYSTERESIS):
		instant_phase = "flexed"
	elif avg_angle > (up_thresh + HYSTERESIS):
		instant_phase = "extended"
	else:
		instant_phase = dumbbell_state["phase"]

	if instant_phase != dumbbell_state["phase"]:
		if dumbbell_state["candidate_phase"] == instant_phase:
			dumbbell_state["candidate_count"] += 1
		else:
			dumbbell_state["candidate_phase"] = instant_phase
			dumbbell_state["candidate_count"] = 1

		if dumbbell_state["candidate_count"] >= STATE_CONFIRM_FRAMES:
			old_phase = dumbbell_state["phase"]
			dumbbell_state["phase"] = instant_phase
			dumbbell_state["candidate_phase"] = None
			dumbbell_state["candidate_count"] = 0

			# ZALICZENIE POWTÓRZENIA: Kiedy wracamy ze stanu ugięcia (flexed) do pełnego wyprostu (extended)
			if old_phase == "flexed" and instant_phase == "extended":
				now = time.time()
				if now - dumbbell_state["last_rep_time"] >= MIN_REP_INTERVAL:
					dumbbell_state["reps"] += 1
					dumbbell_state["last_rep_time"] = now

					current_rep = dumbbell_state["reps"]
					rep_key = f"rep_{current_rep}"
					amplitude = dumbbell_state["cycle_max"] - dumbbell_state["cycle_min"]

					if rep_key not in recording["stats_by_rep"]:
						recording["stats_by_rep"][rep_key] = {"errors": []}

					# Zapisujemy parametry udanego powtórzenia do węzła powtórzenia
					recording["stats_by_rep"][rep_key].update({
						"info": "udane powtorzenie",
						"min_angle": round(dumbbell_state["cycle_min"], 1),
						"max_angle": round(dumbbell_state["cycle_max"], 1),
						"amplitude": round(amplitude, 1)
					})

					if amplitude < exercise["min_amplitude"]:
						recording["stats_by_rep"][rep_key]["errors"].append({
							"error": "Niewystarczajaca amplituda",
							"value": round(amplitude, 1),
							"camera": "both" if both_available else "side"
						})
						speak("wyzej")

					speak(f"powtorzenie {current_rep}")

					# Re-inicjalizacja pod kolejny cykl ruchu
					dumbbell_state["cycle_min"] = avg_angle
					dumbbell_state["cycle_max"] = avg_angle
	else:
		dumbbell_state["candidate_phase"] = None
		dumbbell_state["candidate_count"] = 0

	# --- GRUPOWANIE BŁĘDÓW WSTRZYKIWANYCH DO BIEŻĄCEGO POWTÓRZENIA ---
	current_rep = dumbbell_state["reps"] + 1
	rep_key = f"rep_{current_rep}"

	if recording["active"]:
		if rep_key not in recording["stats_by_rep"]:
			recording["stats_by_rep"][rep_key] = {"errors": []}

		# 1. Asymetria ramion (Front)
		if front_avg is not None:
			try:
				fl, fr, _ = arm_angles(front_landmarks)
				gap = abs(fl - fr)
				if gap > exercise["symmetry_threshold"] and not dumbbell_state.get("symmetry_warned", False):
					recording["stats_by_rep"][rep_key]["errors"].append({
						"error": "Asymetria ramion",
						"value": round(gap, 1),
						"camera": "front"
					})
					dumbbell_state["symmetry_warned"] = True
				elif gap <= exercise["symmetry_threshold"]:
					dumbbell_state["symmetry_warned"] = False
			except Exception:
				pass

		# 2. Bujanie tułowiem (Bok)
		if side_landmarks is not None:
			try:
				shoulder = side_landmarks[11]
				hip = side_landmarks[23]
				dx = shoulder.x - hip.x
				dy = shoulder.y - hip.y
				sway_angle = abs(np.degrees(np.arctan2(dx, -dy)))

				if sway_angle > exercise.get("sway_threshold", 15) and not dumbbell_state.get("sway_warned", False):
					recording["stats_by_rep"][rep_key]["errors"].append({
						"error": "Bujanie tulowiem",
						"value": round(sway_angle, 1),
						"camera": "side"
					})
					speak("nie bujaj")
					dumbbell_state["sway_warned"] = True
				elif sway_angle <= exercise.get("sway_threshold", 15):
					dumbbell_state["sway_warned"] = False
			except Exception:
				pass

		# 3. Uciekający łokieć (Bok)
		if side_landmarks is not None:
			try:
				shoulder = side_landmarks[11]
				elbow = side_landmarks[13]
				hip = side_landmarks[23]
				sway_angle = abs(np.degrees(np.arctan2(shoulder.x - hip.x, -(shoulder.y - hip.y))))

				arm_dx = elbow.x - shoulder.x
				arm_dy = elbow.y - shoulder.y
				arm_angle_vert = abs(np.degrees(np.arctan2(arm_dx, -arm_dy))) - sway_angle

				if abs(arm_angle_vert) > exercise.get("elbow_threshold", 25) and not dumbbell_state.get("elbow_warned", False):
					recording["stats_by_rep"][rep_key]["errors"].append({
						"error": "Uciekajacy lokiec",
						"value": round(abs(arm_angle_vert), 1),
						"camera": "side"
					})
					speak("lokcie blisko")
					dumbbell_state["elbow_warned"] = True
				elif abs(arm_angle_vert) <= exercise.get("elbow_threshold", 25):
					dumbbell_state["elbow_warned"] = False
			except Exception:
				pass

	good = (avg_angle < (down_thresh - HYSTERESIS) + 10) or (avg_angle > (up_thresh + HYSTERESIS) - 10)
	source = "both" if (front_avg is not None and side_avg is not None) else ("side" if side_avg is not None else "front")
	return dumbbell_state["reps"], float(avg_angle), good, source


def save_session(reason="stop"):
	recording["active"] = False
	ts = datetime.now().strftime("%Y%m%d_%H%M%S")
	series_idx = recording.get("series_index", 1)
	Path("sessions").mkdir(exist_ok=True)

	stats_path = f"sessions/stats_{ts}_series_{series_idx}.json"
	video_path = recording.get("video_path")

	if recording.get("video_writer"):
		try:
			recording["video_writer"].release()
		except Exception:
			pass
		recording["video_writer"] = None

	# NOWY STRUKTURYZOWANY PAYLOAD DO PLIKU JSON
	session_payload = {
		"exercise": ACTIVE_EXERCISE_KEY,
		"exercise_label": get_active_exercise()["label"],
		"reps": dumbbell_state.get("reps", 0),
		"reps_detail": recording.get("stats_by_rep", {}),  # Przypisanie ustrukturyzowanych danych
		"series_index": series_idx,
		"reason": reason,
		"auto_save_enabled": save_cfg.get("enabled", True),
		"video_path": video_path,
		"voice_commands": VOICE_COMMANDS,
		"started_at": recording.get("started_at"),
	}

	with open(stats_path, "w", encoding="utf-8") as file:
		json.dump(session_payload, file, indent=2, ensure_ascii=False)

	render_voice("", f"zapisano serie {series_idx}: {dumbbell_state.get('reps',0)} powtorzen")
	recording["series_index"] = series_idx + 1
	recording["frames"] = []
	recording["video_path"] = None
	recording["stats_by_rep"] = {}
	recording["started_at"] = None


def begin_series(reset_reps=True):
	reset_motion_state(reset_reps=reset_reps)
	recording["active"] = True
	recording["frames"] = []
	recording["stats_by_rep"] = {}
	recording["started_at"] = time.time()
	render_status(True)


def end_series(reason="manual"):
	if not recording["active"]:
		return
	save_session(reason=reason)
	render_status(False)


def fetch_camera(camera_name, stream_url, mirror=True):
	cap = cv2.VideoCapture(stream_url)
	cap.set(cv2.CAP_PROP_FPS, FPS)

	thread_options = PoseLandmarkerOptions(
		base_options=BaseOptions(model_asset_path=MODEL_PATH),
		running_mode=VisionRunningMode.VIDEO,
		num_poses=1,
	)

	last_timestamp = 0
	with PoseLandmarker.create_from_options(thread_options) as landmarker:
		while cap.isOpened() and not stop.is_set():
			ret, frame = cap.read()
			if not ret:
				time.sleep(0.03)
				continue

			if mirror:
				frame = cv2.flip(frame, 1)

			rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
			mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

			current_timestamp = int(time.time() * 1000)
			if current_timestamp <= last_timestamp:
				current_timestamp = last_timestamp + 1
			last_timestamp = current_timestamp

			try:
				result = landmarker.detect_for_video(mp_image, current_timestamp)
				landmarks = result.pose_landmarks[0] if result.pose_landmarks else None
			except Exception as e:
				print(f"[MediaPipe Error ({camera_name})]: {e}")
				landmarks = None

			if landmarks is not None:
				color = (0, 255, 136) if camera_name == "front" else (255, 196, 0)
				draw_pose(frame, landmarks, line_color=color)

			with state_lock:
				latest_frames[camera_name] = frame
				latest_landmarks[camera_name] = landmarks
				latest_timestamps[camera_name] = current_timestamp  # Zapis znacznika czasu nowej klatki
	cap.release()


def process_loop():
	while not stop.is_set():
		with state_lock:
			f_ts = latest_timestamps["front"]
			s_ts = latest_timestamps["side"]

			# DEDUPLIKACJA: Jeśli od ostatniego obrotu pętli kamery nie dostarczyły NOWYCH klatek, pomiń krok
			if f_ts == processed_timestamps["front"] and s_ts == processed_timestamps["side"]:
				has_new_data = False
			else:
				has_new_data = True
				processed_timestamps["front"] = f_ts
				processed_timestamps["side"] = s_ts

				front_frame = None if latest_frames["front"] is None else latest_frames["front"].copy()
				side_frame = None if latest_frames["side"] is None else latest_frames["side"].copy()
				front_landmarks = latest_landmarks["front"]
				side_landmarks = latest_landmarks["side"]

		if not has_new_data:
			time.sleep(0.005)
			continue

		reps, avg_angle, good, source = analyze_dumbbell_curl(front_landmarks, side_landmarks)
		render_stats(reps, avg_angle, good, recording["stats_by_rep"], source)

		composed = compose_frames(front_frame, side_frame)
		if composed is None:
			time.sleep(0.03)
			continue

		if recording["active"] and save_cfg.get("enabled", True):
			if recording.get("video_writer") is None:
				h, w = composed.shape[:2]
				ts = datetime.now().strftime("%Y%m%d_%H%M%S")
				series_idx = recording.get("series_index", 1)
				Path("sessions").mkdir(exist_ok=True)
				vp = f"sessions/session_{ts}_series_{series_idx}.avi"
				recording["video_path"] = vp
				fourcc = cv2.VideoWriter_fourcc(*'XVID')
				recording["video_writer"] = cv2.VideoWriter(vp, fourcc, FPS, (w, h))
			try:
				recording["video_writer"].write(composed)
			except Exception:
				pass

		ok, buffer = cv2.imencode(".jpg", composed, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
		if ok:
			img_widget.value = buffer.tobytes()

		time.sleep(0.01)


def _apply_save_toggle(cmd):
	normalized = cmd.strip().lower()
	normalized = re.sub(r"\s+", " ", normalized)
	if "zapis off" in normalized:
		save_cfg["enabled"] = False
		render_status(recording["active"])
		render_voice("", "auto-zapis wylaczony")
		speak("auto zapis off")
		return True
	if "zapis on" in normalized:
		save_cfg["enabled"] = True
		render_status(recording["active"])
		render_voice("", "auto-zapis wlaczony")
		speak("auto zapis on")
		return True
	return False


def stop_everything(reason="stop"):
	if stop.is_set():
		return
	end_series(reason=reason)
	stop.set()


def listen_loop():
    # 1. Tworzymy zamkniętą listę słów dla Vosk (Gramatyka)
    # Dodajemy token "[unk]" dla słów spoza listy, żeby Vosk nie zgadywał na siłę
    all_allowed_words = START_WORDS + STOP_WORDS + SERIES_WORDS + [
        "reset", "zeruj", "zapis on", "zapis off", "[unk]"
    ]
    grammar_json = json.dumps(all_allowed_words, ensure_ascii=False)

    try:
        # Przekazujemy gramatykę jako trzeci parametr - to drastycznie zwiększa celność
        rec = vosk.KaldiRecognizer(vosk_model, 16000, grammar_json)
    except Exception as e:
        print(f"[Vosk Warning] Nie udało się wdrożyć gramatyki, działam w trybie pełnym: {e}")
        rec = vosk.KaldiRecognizer(vosk_model, 16000)

    # 2. Odporna pętla restartująca strumień w razie konfliktu z TTS
    while not stop.is_set():
        try:
            with sd.RawInputStream(samplerate=16000, channels=1, dtype="int16") as stream:
                print("[VOICE] Mikrofon uruchomiony pomyślnie. Nasłuchiwanie...")

                while not stop.is_set():
                    data, _ = stream.read(4000)
                    if len(data) == 0:
                        continue

                    if rec.AcceptWaveform(bytes(data)):
                        res = json.loads(rec.Result())
                        cmd = res.get("text", "").lower().strip()

                        # Ignorujemy puste wyniki oraz szumy zakwalifikowane jako nieznane [unk]
                        if not cmd or cmd == "[unk]":
                            continue

                        render_voice("", cmd)
                        if _apply_save_toggle(cmd):
                            continue

                        normalized = re.sub(r"\s+", " ", cmd)
                        if any(w in normalized for w in START_WORDS):
                            print(f"[VOICE] start detected: {cmd}")
                            begin_series(reset_reps=True)
                            speak("start")
                        elif any(w in normalized for w in SERIES_WORDS):
                            print(f"[VOICE] series delimiter detected: {cmd}")
                            end_series(reason="next_series")
                            begin_series(reset_reps=True)
                            speak("nowa seria")
                        elif "reset" in normalized or "zeruj" in normalized:
                            print(f"[VOICE] reset: {cmd}")
                            reset_motion_state(reset_reps=True)
                            speak("reset")
                        elif any(w in normalized for w in STOP_WORDS):
                            print(f"[VOICE] stop detected: {cmd}")
                            speak(f"koniec powtorzenia {dumbbell_state.get('reps',0)}")
                            stop_everything(reason="voice_stop")
                    else:
                        # Częściowe wyniki (w trakcie mówienia)
                        partial = json.loads(rec.PartialResult()).get("partial", "")
                        if partial and partial != "[unk]":
                            render_voice(partial)

        except Exception as e:
            # Jeśli pyttsx3 odetnie mikrofon, ten blok przechwyci błąd,
            # odczeka chwilę i pętla 'while' spróbuje otworzyć mikrofon ponownie.
            print(f"[Audio Link Glitch] Chwilowa utrata strumienia mikrofonu: {e}")
            time.sleep(0.4)


def start():
	if not Path(MODEL_PATH).exists() or not Path(VOSK_MODEL_PATH).exists():
		print("BLAD: Brak modelu MediaPipe lub Vosk na dysku. Upewnij sie, ze pliki sa we wlasciwym miejscu.")
		return

	stop.clear()
	render_assumptions()
	render_voice("gotowy...")

	reset_motion_state(reset_reps=True)
	render_stats(0, 0, True, {}, "oczekiwanie")

	if AUTO_START_SERIES:
		begin_series(reset_reps=True)
	else:
		render_status(False)

	display(widgets.VBox([
		assumptions_html,
		status_html,
		img_widget,
		stats_html,
		voice_html,
	], layout=widgets.Layout(gap="4px")))

	threading.Thread(target=tts_worker, daemon=True).start()
	threading.Thread(target=fetch_camera, args=("front", FRONT_STREAM_URL, True), daemon=True).start()
	threading.Thread(target=fetch_camera, args=("side", SIDE_STREAM_URL, True), daemon=True).start()
	threading.Thread(target=process_loop, daemon=True).start()
	threading.Thread(target=listen_loop, daemon=True).start()

	if TIMEOUT > 0:
		threading.Timer(TIMEOUT, lambda: stop_everything(reason="timeout")).start()

In [9]:
def playback_session(stats_path, video_path):
	with open(stats_path, encoding="utf-8") as file:
		stats = json.load(file)

	# Pobieramy nową, ustrukturyzowaną formę błędów pogrupowaną po powtórzeniach
	reps_data = stats.get("reps_detail", {})
	cap = cv2.VideoCapture(video_path)
	total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
	cap.release()

	if not total_frames:
		print("Brak klatek w nagraniu.")
		return

	playback_widget = widgets.Image(format="jpeg", layout=widgets.Layout(width="100%"))
	info_html = widgets.HTML()

	def show(idx):
		cap = cv2.VideoCapture(video_path)
		cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
		ret, frame = cap.read()
		cap.release()
		if not ret:
			return

		frame = frame.copy()
		rep_num = max(1, round(idx / max(1, total_frames) * max(1, stats.get("reps", 0))))
		rep_key = f"rep_{rep_num}"

		# Jeśli to powtórzenie miało zarejestrowane jakieś błędy, rysujemy czerwoną ramkę
		if rep_key in reps_data and reps_data[rep_key].get("errors"):
			cv2.rectangle(frame, (0, 0), (frame.shape[1], frame.shape[0]), (0, 0, 255), 6)
			errs = reps_data[rep_key]["errors"]
			cv2.putText(frame, f"BLAD: {errs[0]['error']} ({errs[0]['value']})", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

		ok, buf = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, 75])
		if ok:
			playback_widget.value = buf.tobytes()

		total_errors = sum(len(v.get("errors", [])) for v in reps_data.values())
		info_html.value = (
			f"<div style='font-family:monospace;color:#aaa'>"
			f"klatka {idx + 1}/{total_frames} | powtorzenia: {stats.get('reps', 0)} | "
			f"lacznie bledow: {total_errors} | seria: {stats.get('series_index', '?')}"
			f"</div>"
		)

	slider = widgets.IntSlider(min=0, max=total_frames - 1, step=1, layout=widgets.Layout(width="100%"))
	widgets.interact(show, idx=slider)
	display(widgets.VBox([playback_widget, info_html]))


def playback_latest():
	sessions = sorted(Path("sessions").glob("stats_*_series_*.json"))
	if not sessions:
		print("Brak sesji.")
		return

	stats_path = str(sessions[-1])
	with open(stats_path, encoding="utf-8") as f:
		stats = json.load(f)
	video_path = stats.get("video_path")
	if not video_path or not Path(video_path).exists():
		print("Brak pliku wideo dla tej sesji.")
		return
	playback_session(stats_path, video_path)

In [12]:
start()


In [11]:
#stop.set()

